In [ ]:
import sys, os, warnings
from pathlib import Path
import torch

NOTEBOOK_DIR = Path().resolve()
SRC_DIR      = NOTEBOOK_DIR.parent / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Imports ───────────────────────────────────────────────────────────────────
from dataloaders._ts_dataloader import DataLoaderFactory
from common.train import train
from models.transformer import Transformer
from metrics.torch_losses import get_loss
from _test_utils import make_df, make_mcfg, make_entry, make_dcfg, setup_test_data, remove_test_data

TEST_DATA_DIR = os.path.join(os.getcwd(), "test_data")

warnings.filterwarnings("ignore")
print("imports OK")

## MICA - T5 Encoder 

In [ ]:
try:
    setup_test_data(TEST_DATA_DIR)
    csv_path = f"{TEST_DATA_DIR}/test.csv"
    make_df(n_series=3, n_steps=300).to_csv(csv_path, index=False)

    mcfg = make_mcfg(
        model_name='mica',
        encoder="google/t5-efficient-tiny",
        context_length=32, 
        fcd_samples=4, 
        batch_size=2,
        max_steps=10, 
        checkpoint_dir=TEST_DATA_DIR,
    )
    entry = make_entry(csv_path, name="resume_ds", horizon=6, val_size=50, test_size=50)
    dcfg  = make_dcfg([entry])

    # First run — saves final.pt
    factory      = DataLoaderFactory(mcfg, dcfg)
    train_loader = factory.train_dataloader()
    val_loaders  = factory.val_dataloaders()

    mcfg.loss = get_loss(mcfg.loss)
    loss_fn = mcfg.loss
    model = Transformer(mcfg)
    train(model, mcfg, train_loader, val_loaders, device=torch.device("cpu"), loss_fn=loss_fn)
    step_after_first = model.global_step
    print(f"after first run: global_step={step_after_first}")

    # Second run — resume from final.pt, train 10 more steps
    mcfg2 = make_mcfg(
        model_name='mica',
        encoder="google/t5-efficient-tiny",
        context_length=32, 
        fcd_samples=4, 
        batch_size=2,
        max_steps=step_after_first + 10, 
        checkpoint_dir=TEST_DATA_DIR,
    )
    factory2 = DataLoaderFactory(mcfg2, dcfg)
    model2 = Transformer(mcfg)
    train(
        model2, mcfg2,
        factory2.train_dataloader(), factory2.val_dataloaders(),
        device  = torch.device("cpu"),
        resume  = f"{TEST_DATA_DIR}/final.pt",
        loss_fn=loss_fn
    )
    print(f"after resume:    global_step={model2.global_step}")
    assert model2.global_step == step_after_first + 10, \
        f"expected {step_after_first + 10}, got {model2.global_step}"
    print("\n✓ TEST PASSED")
finally:
    remove_test_data(TEST_DATA_DIR)

## MICA - TST Encoder

In [ ]:
try:
    setup_test_data(TEST_DATA_DIR)
    csv_path = f"{TEST_DATA_DIR}/test.csv"
    make_df(n_series=3, n_steps=300).to_csv(csv_path, index=False)

    mcfg = make_mcfg(
        model_name='mica',
        encoder='patchtst',
        context_length=32, 
        fcd_samples=4, 
        batch_size=2,
        max_steps=10, 
        checkpoint_dir=TEST_DATA_DIR,
    )

    entry = make_entry(csv_path, name="resume_ds", horizon=6, val_size=50, test_size=50)
    dcfg  = make_dcfg([entry])

    # First run — saves final.pt
    factory      = DataLoaderFactory(mcfg, dcfg)
    train_loader = factory.train_dataloader()
    val_loaders  = factory.val_dataloaders()

    mcfg.loss = get_loss(mcfg.loss)
    loss_fn = mcfg.loss
    model = Transformer(mcfg)
    train(model, mcfg, train_loader, val_loaders, device=torch.device("cpu"), loss_fn=loss_fn)
    step_after_first = model.global_step
    print(f"after first run: global_step={step_after_first}")

    # Second run — resume from final.pt, train 10 more steps
    mcfg2 = make_mcfg(
        model_name='mica',
        encoder='patchtst',
        context_length=32, 
        fcd_samples=4, 
        batch_size=2,
        max_steps=step_after_first + 10, 
        checkpoint_dir=TEST_DATA_DIR,
    )

    factory2 = DataLoaderFactory(mcfg2, dcfg)
    model2 = Transformer(mcfg)
    train(
        model2, mcfg2,
        factory2.train_dataloader(), factory2.val_dataloaders(),
        device  = torch.device("cpu"),
        resume  = f"{TEST_DATA_DIR}/final.pt",
        loss_fn=loss_fn
    )
    print(f"after resume:    global_step={model2.global_step}")
    assert model2.global_step == step_after_first + 10, \
        f"expected {step_after_first + 10}, got {model2.global_step}"
    print("\n✓ TEST PASSED")
finally:
    remove_test_data(TEST_DATA_DIR)